# Import libraries

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Configuration

In [2]:
# Tìm project root: đi ngược từ thư mục hiện tại đến khi gặp pyproject.toml + data/
current_dir = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [current_dir, *current_dir.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "data").is_dir():
        PROJECT_ROOT = candidate
        break

# Dự phòng: notebook nằm trong notebooks/ thì root là thư mục cha
if PROJECT_ROOT is None:
    PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "uci" / "online_retail_II.xlsx"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH   :", DATA_PATH.relative_to(PROJECT_ROOT))
print("Exists      :", DATA_PATH.exists())

PROJECT_ROOT: D:\data science\nexora-commerce-ai\nexora-commerce-ai
DATA_PATH   : data\raw\uci\online_retail_II.xlsx
Exists      : True


In [3]:
# Dừng sớm nếu không có file raw
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy dataset tại: {DATA_PATH}\n"
        "Hãy đặt file online_retail_II.xlsx vào data/raw/uci/ rồi chạy lại."
    )

file_size_mb = DATA_PATH.stat().st_size / (1024 ** 2)
print(f"File size: {file_size_mb:.2f} MB")

File size: 43.51 MB


In [4]:
# Hiển thị bảng rộng hơn cho dễ đọc
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

RANDOM_STATE = 42

# Load raw data

In [5]:
# Đọc metadata trước để biết tên sheet thực tế (không giả định)
excel_file = pd.ExcelFile(DATA_PATH)
sheet_names = excel_file.sheet_names

print("Number of sheets:", len(sheet_names))
for name in sheet_names:
    print(" -", repr(name))

Number of sheets: 2
 - 'Year 2009-2010'
 - 'Year 2010-2011'


In [6]:
# Đọc từng sheet thành DataFrame riêng (chỉ đọc, không ghi lại file)
# Lưu ý: file Excel lớn nên bước này có thể mất vài phút
raw_sheets = {}
for name in sheet_names:
    raw_sheets[name] = pd.read_excel(excel_file, sheet_name=name)
    print(f"{name!r}: shape = {raw_sheets[name].shape}")

'Year 2009-2010': shape = (525461, 8)
'Year 2010-2011': shape = (541910, 8)


In [7]:
# Kiểm tra tên cột của từng sheet
for name, frame in raw_sheets.items():
    print(f"{name!r}: {list(frame.columns)}")

# Schema tương thích khi tất cả sheet có cùng danh sách cột
column_sets = [tuple(frame.columns) for frame in raw_sheets.values()]
schemas_compatible = len(set(column_sets)) == 1
print("\nSchemas compatible:", schemas_compatible)

'Year 2009-2010': ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
'Year 2010-2011': ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Schemas compatible: True


In [8]:
# Tạo DataFrame tổng hợp chỉ phục vụ audit, có thêm cột source_sheet
# assign() trả về bản copy nên raw_sheets không bị thay đổi
if schemas_compatible:
    audit_df = pd.concat(
        [frame.assign(source_sheet=name) for name, frame in raw_sheets.items()],
        ignore_index=True,
    )
else:
    # Nếu schema khác nhau thì chỉ audit sheet đầu tiên để tránh ghép sai cột
    print("Schema không tương thích -> chỉ dùng sheet đầu tiên cho audit tổng hợp.")
    first_sheet = sheet_names[0]
    audit_df = raw_sheets[first_sheet].assign(source_sheet=first_sheet)

# Các cột gốc (không tính cột source_sheet thêm vào)
data_columns = [c for c in audit_df.columns if c != "source_sheet"]
print("Combined audit shape:", audit_df.shape)

Combined audit shape: (1067371, 9)


In [9]:
# Helper: tìm tên cột thực tế theo danh sách tên ứng viên (không phân biệt hoa/thường, khoảng trắng)
def find_column(frame, candidates):
    normalized = {str(c).strip().lower().replace(" ", ""): c for c in frame.columns}
    for cand in candidates:
        key = cand.strip().lower().replace(" ", "")
        if key in normalized:
            return normalized[key]
    return None


INVOICE_COL = find_column(audit_df, ["Invoice", "InvoiceNo"])
STOCK_COL = find_column(audit_df, ["StockCode", "ProductCode"])
DESC_COL = find_column(audit_df, ["Description"])
QTY_COL = find_column(audit_df, ["Quantity"])
DATE_COL = find_column(audit_df, ["InvoiceDate", "Date"])
PRICE_COL = find_column(audit_df, ["Price", "UnitPrice"])
CUSTOMER_COL = find_column(audit_df, ["Customer ID", "CustomerID"])
COUNTRY_COL = find_column(audit_df, ["Country"])

resolved_columns = {
    "invoice": INVOICE_COL, "stock_code": STOCK_COL, "description": DESC_COL,
    "quantity": QTY_COL, "date": DATE_COL, "price": PRICE_COL,
    "customer": CUSTOMER_COL, "country": COUNTRY_COL,
}
# None nghĩa là không tìm thấy cột -> các check liên quan sẽ bị bỏ qua
resolved_columns

{'invoice': 'Invoice',
 'stock_code': 'StockCode',
 'description': 'Description',
 'quantity': 'Quantity',
 'date': 'InvoiceDate',
 'price': 'Price',
 'customer': 'Customer ID',
 'country': 'Country'}

# Dataset overview

In [10]:
# Summary tổng quan (không tính cột source_sheet)
overview = pd.Series(
    {
        "number_of_rows": len(audit_df),
        "number_of_columns": len(data_columns),
        "number_of_sheets": len(raw_sheets),
    },
    name="value",
)
overview.to_frame()

,value
number_of_rows,1067371
number_of_columns,8
number_of_sheets,2


In [11]:
# Shape từng sheet và shape tổng
sheet_shapes = pd.DataFrame(
    [{"sheet": name, "rows": f.shape[0], "columns": f.shape[1]} for name, f in raw_sheets.items()]
)
display(sheet_shapes)
print("Combined shape (gồm source_sheet):", audit_df.shape)
print("Columns:", list(audit_df.columns))

,sheet,rows,columns
0,Year 2009-2010,525461,8
1,Year 2010-2011,541910,8


Combined shape (gồm source_sheet): (1067371, 9)
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'source_sheet']


In [12]:
# 5 dòng đầu
audit_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


In [13]:
# 5 dòng cuối
audit_df.tail()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,Year 2010-2011
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,Year 2010-2011
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,Year 2010-2011
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France,Year 2010-2011
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680.0,France,Year 2010-2011


In [14]:
# Sample ngẫu nhiên với random_state cố định
audit_df.sample(10, random_state=RANDOM_STATE)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
455941,532657,21314,SMALL GLASS HEART TRINKET POT,12,2010-11-14 11:10:00,2.10,14562.0,United Kingdom,Year 2009-2010
826291,563214,22383,LUNCH BAG SUKI DESIGN,2,2011-08-14 12:56:00,1.65,16370.0,United Kingdom,Year 2010-2011
191636,507597,22561,WOODEN SCHOOL COLOURING SET,12,2010-05-10 13:21:00,1.65,17700.0,United Kingdom,Year 2009-2010
25864,491634,21588,RETRO SPOT GIANT TUBE MATCHES,1,2009-12-11 15:40:00,2.55,17841.0,United Kingdom,Year 2009-2010
73233,496007,85232B,SET/3 RUSSIAN DOLL STACKING TINS,3,2010-01-28 12:32:00,4.95,15203.0,United Kingdom,Year 2009-2010
557542,539041,21832,CHOCOLATE CALCULATOR,4,2010-12-15 15:34:00,1.65,15456.0,United Kingdom,Year 2010-2011
985772,575905,22089,PAPER BUNTING VINTAGE PAISLEY,6,2011-11-11 15:49:00,2.95,13732.0,United Kingdom,Year 2010-2011
568585,540026,21519,GIN & TONIC DIET GREETING CARD,2,2011-01-04 13:25:00,0.85,NaN,United Kingdom,Year 2010-2011
530265,536804,22988,SOLDIERS EGG CUP,72,2010-12-02 16:34:00,1.25,14031.0,United Kingdom,Year 2010-2011
649024,546899,20719,WOODLAND CHARLOTTE BAG,50,2011-03-17 18:27:00,0.72,14298.0,United Kingdom,Year 2010-2011


In [15]:
audit_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   Invoice       1067371 non-null  object        
 1   StockCode     1067371 non-null  object        
 2   Description   1062989 non-null  object        
 3   Quantity      1067371 non-null  int64         
 4   InvoiceDate   1067371 non-null  datetime64[us]
 5   Price         1067371 non-null  float64       
 6   Customer ID   824364 non-null   float64       
 7   Country       1067371 non-null  str           
 8   source_sheet  1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(2)
memory usage: 73.3+ MB


# Data types

In [16]:
# dtype, số giá trị unique và ví dụ giá trị của từng cột
dtype_summary = pd.DataFrame(
    {
        "dtype": audit_df.dtypes.astype(str),
        "unique_count": audit_df.nunique(dropna=True),
        "example_values": [audit_df[c].dropna().unique()[:5].tolist() for c in audit_df.columns],
    }
)
dtype_summary

,dtype,unique_count,example_values
Invoice,object,53628,"[489434, 489435, 489436, 489437, 489438]"
StockCode,object,5305,"[85048, 79323P, 79323W, 22041, 21232]"
Description,object,5698,"[15CM CHRISTMAS GLASS BALL 20 LIGHTS, PINK CHE..."
Quantity,int64,1057,"[12, 48, 24, 10, 18]"
InvoiceDate,datetime64[us],47635,"[2009-12-01 07:45:00, 2009-12-01 07:46:00, 200..."
Price,float64,2807,"[6.95, 6.75, 2.1, 1.25, 1.65]"
Customer ID,float64,5942,"[13085.0, 13078.0, 15362.0, 18102.0, 12682.0]"
Country,str,43,"[United Kingdom, France, USA, Belgium, Australia]"
source_sheet,str,2,"[Year 2009-2010, Year 2010-2011]"


In [17]:
# Nhóm cột theo kiểu dữ liệu
datetime_columns = audit_df.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()
numeric_columns = audit_df.select_dtypes(include="number").columns.tolist()
text_columns = [c for c in audit_df.columns if c not in datetime_columns + numeric_columns]

print("Datetime columns   :", datetime_columns)
print("Numeric columns    :", numeric_columns)
print("Object/string cols :", text_columns)

Datetime columns   : ['InvoiceDate']
Numeric columns    : ['Quantity', 'Price', 'Customer ID']
Object/string cols : ['Invoice', 'StockCode', 'Description', 'Country', 'source_sheet']


In [18]:
# Kiểm tra cột text có lẫn nhiều kiểu Python (vd: int lẫn str) -> dấu hiệu dtype đáng nghi
mixed_type_rows = []
for col in text_columns:
    type_counts = audit_df[col].dropna().map(lambda v: type(v).__name__).value_counts()
    for type_name, count in type_counts.items():
        mixed_type_rows.append({"column": col, "python_type": type_name, "count": count})

pd.DataFrame(mixed_type_rows)

,column,python_type,count
0,Invoice,int,1047871
1,Invoice,str,19500
2,StockCode,int,932385
3,StockCode,str,134986
4,Description,str,1062985
5,Description,int,4
6,Country,str,1067371
7,source_sheet,str,1067371


In [19]:
# Customer ID: kiểm tra có bị đọc thành float không và giá trị có phải số nguyên không
# Chỉ ghi nhận, KHÔNG convert ở bước audit
if CUSTOMER_COL is not None:
    customer_values = audit_df[CUSTOMER_COL].dropna()
    print("dtype:", audit_df[CUSTOMER_COL].dtype)
    if pd.api.types.is_numeric_dtype(customer_values):
        non_integer = (customer_values % 1 != 0).sum()
        print("Non-null values:", len(customer_values))
        print("Non-integer values:", int(non_integer))
        print("-> Nếu dtype là float do có missing, cần chuyển sang kiểu ID (Int64/string) ở cleaning stage.")
else:
    print("Không tìm thấy cột customer.")

dtype: float64
Non-null values: 824364
Non-integer values: 0
-> Nếu dtype là float do có missing, cần chuyển sang kiểu ID (Int64/string) ở cleaning stage.


# Missing values

In [20]:
# Helper: bảng missing count / percent cho một DataFrame
def missing_table(frame, columns):
    missing_count = frame[columns].isna().sum()
    return (
        pd.DataFrame(
            {
                "column": missing_count.index,
                "missing_count": missing_count.values,
                "missing_percent": (missing_count.values / len(frame) * 100).round(3),
            }
        )
        .sort_values("missing_count", ascending=False)
        .reset_index(drop=True)
    )


missing_summary = missing_table(audit_df, data_columns)
# Chỉ hiển thị các cột có missing
missing_summary[missing_summary["missing_count"] > 0]

,column,missing_count,missing_percent
0,Customer ID,243007,22.767
1,Description,4382,0.411


In [21]:
# Missing theo từng sheet
missing_by_sheet = []
for name, frame in raw_sheets.items():
    table = missing_table(frame, list(frame.columns))
    table.insert(0, "sheet", name)
    missing_by_sheet.append(table[table["missing_count"] > 0])

pd.concat(missing_by_sheet, ignore_index=True)

,sheet,column,missing_count,missing_percent
0,Year 2009-2010,Customer ID,107927,20.539
1,Year 2009-2010,Description,2928,0.557
2,Year 2010-2011,Customer ID,135080,24.927
3,Year 2010-2011,Description,1454,0.268


In [22]:
# Chú ý riêng các trường nhận diện customer / product / invoice
key_columns = [c for c in [CUSTOMER_COL, STOCK_COL, DESC_COL, INVOICE_COL] if c is not None]
missing_summary[missing_summary["column"].isin(key_columns)]

,column,missing_count,missing_percent
0,Customer ID,243007,22.767
1,Description,4382,0.411
2,StockCode,0,0.000
3,Invoice,0,0.000


In [23]:
# Missing Description có đi kèm missing Customer ID không? (chỉ quan sát, không fill)
if DESC_COL is not None and CUSTOMER_COL is not None:
    display(
        pd.crosstab(
            audit_df[DESC_COL].isna().rename("description_missing"),
            audit_df[CUSTOMER_COL].isna().rename("customer_missing"),
        )
    )
    if PRICE_COL is not None:
        print("Price của các dòng thiếu Description:")
        display(audit_df.loc[audit_df[DESC_COL].isna(), PRICE_COL].describe())

customer_missing,False,True
description_missing,,
False,824364,238625
True,0,4382


Price của các dòng thiếu Description:


count    4382.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: Price, dtype: float64

# Duplicate analysis

In [24]:
# Duplicate toàn bộ row trên các cột gốc (bỏ qua source_sheet)
dup_mask_all = audit_df.duplicated(subset=data_columns, keep=False)  # mọi dòng thuộc nhóm trùng
dup_mask_extra = audit_df.duplicated(subset=data_columns, keep="first")  # bản sao "dư" sau dòng đầu

duplicate_summary = pd.Series(
    {
        "rows_in_duplicate_groups": int(dup_mask_all.sum()),
        "duplicate_count (extra copies)": int(dup_mask_extra.sum()),
        "duplicate_percent": round(dup_mask_extra.mean() * 100, 3),
    },
    name="combined",
)
duplicate_summary.to_frame()

,combined
rows_in_duplicate_groups,67242.000
duplicate_count (extra copies),34335.000
duplicate_percent,3.217


In [25]:
# Duplicate trong từng sheet và duplicate xuất hiện ở nhiều sheet
dup_rows = []
for name, frame in raw_sheets.items():
    n_dup = int(frame.duplicated().sum())
    dup_rows.append({"scope": f"within {name!r}", "duplicate_count": n_dup, "duplicate_percent": round(n_dup / len(frame) * 100, 3)})

within_sheet_total = sum(r["duplicate_count"] for r in dup_rows)
dup_rows.append({"scope": "cross-sheet only (combined - within)", "duplicate_count": int(dup_mask_extra.sum()) - within_sheet_total, "duplicate_percent": None})
pd.DataFrame(dup_rows)

,scope,duplicate_count,duplicate_percent
0,within 'Year 2009-2010',6865,1.306
1,within 'Year 2010-2011',5268,0.972
2,cross-sheet only (combined - within),22202,NaN


In [26]:
# Sample các dòng trùng, sắp xếp để các bản sao nằm cạnh nhau
if dup_mask_all.any():
    sort_cols = [c for c in [INVOICE_COL, STOCK_COL, DATE_COL] if c is not None]
    display(audit_df.loc[dup_mask_all].sort_values(sort_cols).head(12))
else:
    print("Không phát hiện duplicated rows.")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,Year 2009-2010
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,Year 2009-2010
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom,Year 2009-2010


> **Lưu ý:** Các dòng giống hoàn toàn chưa chắc luôn là lỗi nghiệp vụ. Dataset không có line-level unique identifier
> (chỉ có invoice + stock code), nên một khách có thể thật sự nhập cùng sản phẩm nhiều lần trong cùng invoice.
> Ngoài ra, duplicate giữa các sheet có thể đến từ việc các sheet chồng lấn khoảng thời gian (xem phần Date analysis).
> Notebook này **không** `drop_duplicates`.

# Invalid values

In [27]:
# Thống kê mô tả cho numeric columns
audit_df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
Quantity,1067371.0,9.938898,172.705794,-80995.00,1.00,3.0,10.00,80995.0
Price,1067371.0,4.649388,123.553059,-53594.36,1.25,2.1,4.15,38970.0
Customer ID,824364.0,15324.638504,1697.464450,12346.00,13975.00,15255.0,16797.00,18287.0


In [28]:
# Phân vị để xem mức độ bất thường của Quantity / Price
quantile_levels = [0.0, 0.001, 0.01, 0.25, 0.5, 0.75, 0.99, 0.999, 1.0]
value_columns = [c for c in [QTY_COL, PRICE_COL] if c is not None]
audit_df[value_columns].quantile(quantile_levels)

,Quantity,Price
0.000,-80995.00,-53594.360
0.001,-112.63,0.000
0.010,-3.00,0.210
0.250,1.00,1.250
0.500,3.00,2.100
0.750,10.00,4.150
0.990,100.00,18.000
0.999,500.00,217.016
1.000,80995.00,38970.000


In [29]:
# Tạo mask cho từng loại vấn đề - chỉ check khi cột thực sự tồn tại
issue_masks = {}

if QTY_COL is not None:
    qty = audit_df[QTY_COL]
    issue_masks[f"{QTY_COL} <= 0"] = qty <= 0
    issue_masks[f"{QTY_COL} < 0"] = qty < 0
    issue_masks[f"{QTY_COL} == 0"] = qty == 0
    # Ngưỡng "cực lớn" tham khảo: |Quantity| vượt phân vị 99.9% của |Quantity|
    qty_threshold = qty.abs().quantile(0.999)
    issue_masks[f"|{QTY_COL}| > p99.9 ({qty_threshold:g})"] = qty.abs() > qty_threshold

if PRICE_COL is not None:
    price = audit_df[PRICE_COL]
    issue_masks[f"{PRICE_COL} <= 0"] = price <= 0
    issue_masks[f"{PRICE_COL} < 0"] = price < 0
    issue_masks[f"{PRICE_COL} == 0"] = price == 0
    price_threshold = price.quantile(0.999)
    issue_masks[f"{PRICE_COL} > p99.9 ({price_threshold:g})"] = price > price_threshold

# Các trường nhận diện bị thiếu
for label, col in [("stock code missing", STOCK_COL), ("invoice missing", INVOICE_COL),
                   ("customer ID missing", CUSTOMER_COL), ("description missing", DESC_COL)]:
    if col is not None:
        issue_masks[label] = audit_df[col].isna()

# Description rỗng sau khi bỏ khoảng trắng (khác với NaN)
if DESC_COL is not None:
    desc_text = audit_df[DESC_COL].dropna().astype(str).str.strip()
    issue_masks["description blank string"] = audit_df.index.isin(desc_text[desc_text == ""].index)

invalid_summary = pd.DataFrame(
    [{"issue": k, "count": int(m.sum()), "percent": round(float(np.mean(m)) * 100, 3)} for k, m in issue_masks.items()]
)
invalid_summary

,issue,count,percent
0,Quantity <= 0,22950,2.150
1,Quantity < 0,22950,2.150
2,Quantity == 0,0,0.000
3,|Quantity| > p99.9 (600),941,0.088
4,Price <= 0,6207,0.582
5,Price < 0,5,0.000
6,Price == 0,6202,0.581
7,Price > p99.9 (217.016),1068,0.100
8,stock code missing,0,0.000
9,invoice missing,0,0.000


In [30]:
# Các dòng có |Quantity| lớn nhất
if QTY_COL is not None:
    display(audit_df.loc[audit_df[QTY_COL].abs().nlargest(10).index])

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,Year 2010-2011
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom,Year 2010-2011
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,Year 2010-2011
587085,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346.0,United Kingdom,Year 2010-2011
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2010-02-15 11:57:00,0.10,13902.0,Denmark,Year 2009-2010
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark,Year 2009-2010
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark,Year 2009-2010
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,2010-03-17 13:09:00,0.10,13902.0,Denmark,Year 2009-2010
1027583,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.00,13256.0,United Kingdom,Year 2010-2011
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,2010-03-17 13:09:00,0.10,13902.0,Denmark,Year 2009-2010


In [31]:
# Các dòng có Price lớn nhất
if PRICE_COL is not None:
    display(audit_df.nlargest(10, PRICE_COL))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
748142,C556445,M,Manual,-1,2011-06-10 15:31:00,38970.00,15098.0,United Kingdom,Year 2010-2011
241824,C512770,M,Manual,-1,2010-06-17 16:52:00,25111.09,17399.0,United Kingdom,Year 2009-2010
241827,512771,M,Manual,1,2010-06-17 16:53:00,25111.09,NaN,United Kingdom,Year 2009-2010
320581,C520667,BANK CHARGES,Bank Charges,-1,2010-08-27 13:42:00,18910.69,NaN,United Kingdom,Year 2009-2010
1050063,C580605,AMAZONFEE,AMAZON FEE,-1,2011-12-05 11:36:00,17836.46,NaN,United Kingdom,Year 2010-2011
569163,C540117,AMAZONFEE,AMAZON FEE,-1,2011-01-05 09:55:00,16888.02,NaN,United Kingdom,Year 2010-2011
569164,C540118,AMAZONFEE,AMAZON FEE,-1,2011-01-05 09:57:00,16453.71,NaN,United Kingdom,Year 2010-2011
517953,C537630,AMAZONFEE,AMAZON FEE,-1,2010-12-07 15:04:00,13541.33,NaN,United Kingdom,Year 2009-2010
517955,537632,AMAZONFEE,AMAZON FEE,1,2010-12-07 15:08:00,13541.33,NaN,United Kingdom,Year 2009-2010
519294,C537651,AMAZONFEE,AMAZON FEE,-1,2010-12-07 15:49:00,13541.33,NaN,United Kingdom,Year 2009-2010


In [32]:
# Các dòng có Price âm (nếu có)
if PRICE_COL is not None:
    negative_price = audit_df[audit_df[PRICE_COL] < 0]
    print("Rows with negative price:", len(negative_price))
    display(negative_price.head(10))

Rows with negative price: 5


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom,Year 2009-2010
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom,Year 2009-2010
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom,Year 2009-2010
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,Year 2010-2011
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,Year 2010-2011


In [33]:
# Stock code không theo dạng bắt đầu bằng 5 chữ số -> có thể là mã phi sản phẩm (phí, điều chỉnh...)
# Chỉ quan sát pattern, chưa kết luận
if STOCK_COL is not None:
    stock_text = audit_df[STOCK_COL].astype(str).str.strip()
    non_standard_stock = ~stock_text.str.match(r"^\d{5}")
    print("Rows with non-standard stock code:", int(non_standard_stock.sum()))
    print("Unique non-standard stock codes:", stock_text[non_standard_stock].nunique())

    top_codes = stock_text[non_standard_stock].value_counts().head(20).rename("row_count").to_frame()
    if DESC_COL is not None:
        # Gắn 1 description mẫu cho mỗi mã để dễ hiểu
        sample_desc = audit_df.loc[non_standard_stock].groupby(stock_text[non_standard_stock])[DESC_COL].first()
        top_codes["sample_description"] = sample_desc.reindex(top_codes.index)
    display(top_codes)

Rows with non-standard stock code: 6093
Unique non-standard stock codes: 62


,row_count,sample_description
StockCode,,
POST,2122,POSTAGE
DOT,1446,DOTCOM POSTAGE
M,1421,Manual
C2,282,CARRIAGE
D,177,Discount
S,104,SAMPLES
BANK CHARGES,102,Bank Charges
ADJUST,67,Adjustment by john on 26/01/2010 16
AMAZONFEE,43,AMAZON FEE


# Returns / cancellations

In [34]:
# Kiểm tra pattern của invoice: tách phần prefix chữ cái ở đầu (nếu có)
invoice_text = audit_df[INVOICE_COL].astype(str).str.strip()
invoice_prefix = invoice_text.str.extract(r"^([A-Za-z]+)", expand=False).fillna("<numeric>")

prefix_summary = pd.DataFrame(
    {
        "row_count": invoice_prefix.value_counts(),
        "unique_invoices": invoice_text.groupby(invoice_prefix).nunique(),
    }
)
prefix_summary

,row_count,unique_invoices
Invoice,,
<numeric>,1047871,45330
A,6,6
C,19494,8292


In [35]:
# Quantity âm
negative_qty_mask = audit_df[QTY_COL] < 0
print("Rows with negative quantity:", int(negative_qty_mask.sum()))
print(f"Percent of rows: {negative_qty_mask.mean() * 100:.3f}%")

Rows with negative quantity: 22950
Percent of rows: 2.150%


In [36]:
# Chỉ đo cancellation theo prefix "C" nếu pattern này thực sự xuất hiện
has_c_prefix = (invoice_prefix == "C").any()
print("Invoice prefix 'C' present:", has_c_prefix)

if has_c_prefix:
    cancel_mask = invoice_prefix == "C"
    cancel_summary = pd.Series(
        {
            "cancellation_invoices": invoice_text[cancel_mask].nunique(),
            "total_invoices": invoice_text.nunique(),
            "cancellation_invoice_percent": round(invoice_text[cancel_mask].nunique() / invoice_text.nunique() * 100, 3),
            "cancellation_rows": int(cancel_mask.sum()),
            "cancellation_row_percent": round(cancel_mask.mean() * 100, 3),
        },
        name="value",
    )
    display(cancel_summary.to_frame())

Invoice prefix 'C' present: True


,value
cancellation_invoices,8292.000
total_invoices,53628.000
cancellation_invoice_percent,15.462
cancellation_rows,19494.000
cancellation_row_percent,1.826


In [37]:
# Mối liên hệ giữa invoice cancellation và quantity âm
if has_c_prefix:
    qty_sign = np.sign(audit_df[QTY_COL]).map({-1: "negative", 0: "zero", 1: "positive"})
    display(pd.crosstab(cancel_mask.rename("is_C_invoice"), qty_sign.rename("quantity_sign"), margins=True))

quantity_sign,negative,positive,All
is_C_invoice,,,
False,3457,1044420,1047877
True,19493,1,19494
All,22950,1044421,1067371


In [38]:
# Sample: invoice cancellation
if has_c_prefix:
    display(audit_df[cancel_mask].sample(min(8, int(cancel_mask.sum())), random_state=RANDOM_STATE))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
908262,C569955,23110,PARISIENNE KEY CABINET,-2,2011-10-06 18:35:00,5.75,17644.0,United Kingdom,Year 2010-2011
236032,C512272,22027,TEA PARTY BIRTHDAY CARD,-12,2010-06-14 13:32:00,0.42,15031.0,United Kingdom,Year 2009-2010
828392,C563483,22927,GREEN GIANT GARDEN THERMOMETER,-1,2011-08-16 15:59:00,5.95,14175.0,United Kingdom,Year 2010-2011
626165,C544828,84279B,CHERRY BLOSSOM DECORATIVE FLASK,-1,2011-02-24 09:49:00,3.75,17147.0,United Kingdom,Year 2010-2011
45469,C493440,21843,RETRO SPOT CAKE STAND,-1,2010-01-04 13:30:00,9.95,12709.0,Germany,Year 2009-2010
39047,C492673,21929,JUMBO BAG PINK VINTAGE PAISLEY,-1,2009-12-17 18:13:00,1.95,13141.0,United Kingdom,Year 2009-2010
258902,C514368,84849D,HOT BATHS SOAP HOLDER,-2,2010-07-01 19:19:00,1.69,15015.0,United Kingdom,Year 2009-2010
82928,C497003,21217,RED SPOTTY ROUND CAKE TINS,-2,2010-02-05 12:00:00,9.95,15465.0,United Kingdom,Year 2009-2010


In [39]:
# Sample: Quantity âm nhưng KHÔNG phải invoice "C" -> ý nghĩa có thể khác (điều chỉnh kho, hàng hỏng...)
other_negative = negative_qty_mask & ~(invoice_prefix == "C")
print("Negative quantity rows without 'C' prefix:", int(other_negative.sum()))
if other_negative.any():
    display(audit_df[other_negative].sample(min(8, int(other_negative.sum())), random_state=RANDOM_STATE))
    if PRICE_COL is not None:
        print("Price của nhóm này:")
        display(audit_df.loc[other_negative, PRICE_COL].describe())
    if CUSTOMER_COL is not None:
        print("Missing customer trong nhóm này:", int(audit_df.loc[other_negative, CUSTOMER_COL].isna().sum()))

Negative quantity rows without 'C' prefix: 3457


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
407070,528371,22275,NaN,-34,2010-10-21 16:51:00,0.0,NaN,United Kingdom,Year 2009-2010
32108,491978,72755C,NaN,-60,2009-12-14 18:45:00,0.0,NaN,United Kingdom,Year 2009-2010
68503,495412,72715,NaN,-30,2010-01-25 11:11:00,0.0,NaN,United Kingdom,Year 2009-2010
727302,554328,21609,NaN,-46,2011-05-23 16:49:00,0.0,NaN,United Kingdom,Year 2010-2011
472120,533868,21843,damaged,-25,2010-11-19 11:34:00,0.0,NaN,United Kingdom,Year 2009-2010
662701,548149,72821,NaN,-2,2011-03-29 12:36:00,0.0,NaN,United Kingdom,Year 2010-2011
532666,537013,35965,NaN,-25,2010-12-03 15:40:00,0.0,NaN,United Kingdom,Year 2010-2011
657160,547593,20785,NaN,-2,2011-03-24 11:18:00,0.0,NaN,United Kingdom,Year 2010-2011


Price của nhóm này:


count    3457.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: Price, dtype: float64

Missing customer trong nhóm này: 3457


In [40]:
# Sample các invoice prefix khác ngoài "C" (nếu có)
other_prefixes = [p for p in prefix_summary.index if p not in ("<numeric>", "C")]
print("Other invoice prefixes:", other_prefixes)
if other_prefixes:
    display(audit_df[invoice_prefix.isin(other_prefixes)].head(10))

Other invoice prefixes: ['A']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom,Year 2009-2010
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom,Year 2009-2010
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom,Year 2009-2010
825443,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom,Year 2010-2011
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,Year 2010-2011
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,Year 2010-2011


# Date analysis

In [41]:
# Xác định cột ngày thực tế và kiểm tra dtype (không chỉnh sửa timestamp)
if DATE_COL is None and datetime_columns:
    DATE_COL = datetime_columns[0]

print("Date column:", DATE_COL)
print("dtype:", audit_df[DATE_COL].dtype)
print("Is datetime dtype:", pd.api.types.is_datetime64_any_dtype(audit_df[DATE_COL]))
print("Missing dates:", int(audit_df[DATE_COL].isna().sum()))

Date column: InvoiceDate
dtype: datetime64[us]
Is datetime dtype: True
Missing dates: 0


In [42]:
dates = audit_df[DATE_COL]
date_overview = pd.Series(
    {
        "min_date": dates.min(),
        "max_date": dates.max(),
        "calendar_span_days": (dates.max().normalize() - dates.min().normalize()).days + 1,
        "observed_days (có giao dịch)": dates.dt.normalize().nunique(),
    },
    name="value",
)
date_overview.to_frame()

,value
min_date,2009-12-01 07:45:00
max_date,2011-12-09 12:50:00
calendar_span_days,739
observed_days (có giao dịch),604


In [43]:
# Số record theo năm
dates.dt.year.value_counts().sort_index().rename("record_count").to_frame()

,record_count
InvoiceDate,
2009,45228
2010,522714
2011,499429


In [44]:
# Số record và số ngày có giao dịch theo tháng
monthly = pd.DataFrame(
    {
        "record_count": dates.groupby(dates.dt.to_period("M")).size(),
        "active_days": dates.dt.normalize().groupby(dates.dt.to_period("M")).nunique(),
        "first_date": dates.groupby(dates.dt.to_period("M")).min(),
        "last_date": dates.groupby(dates.dt.to_period("M")).max(),
    }
)
monthly

,record_count,active_days,first_date,last_date
InvoiceDate,,,,
2009-12,45228,21,2009-12-01 07:45:00,2009-12-23 16:58:00
2010-01,31555,24,2010-01-04 09:24:00,2010-01-31 16:02:00
2010-02,29388,24,2010-02-01 08:13:00,2010-02-28 16:16:00
2010-03,41511,27,2010-03-01 08:38:00,2010-03-31 17:53:00
2010-04,34057,23,2010-04-01 07:49:00,2010-04-30 17:33:00
2010-05,35323,24,2010-05-02 10:22:00,2010-05-30 15:58:00
2010-06,39983,26,2010-06-01 09:02:00,2010-06-30 17:38:00
2010-07,33383,26,2010-07-01 08:39:00,2010-07-30 17:08:00
2010-08,33306,26,2010-08-01 10:04:00,2010-08-31 17:44:00


In [45]:
# Tháng đầu kỳ / cuối kỳ có đủ dữ liệu không? So ngày đầu/cuối quan sát với ngày đầu/cuối tháng
first_month, last_month = monthly.index.min(), monthly.index.max()
print(f"First month {first_month}: data from {monthly.loc[first_month, 'first_date']} "
      f"(month starts {first_month.start_time.date()})")
print(f"Last month  {last_month}: data until {monthly.loc[last_month, 'last_date']} "
      f"(month ends {last_month.end_time.date()})")

# Kiểm tra có tháng nào bị thiếu hoàn toàn trong khoảng thời gian không
all_months = pd.period_range(first_month, last_month, freq="M")
missing_months = all_months.difference(monthly.index)
print("Missing months in range:", list(missing_months.astype(str)))

First month 2009-12: data from 2009-12-01 07:45:00 (month starts 2009-12-01)
Last month  2011-12: data until 2011-12-09 12:50:00 (month ends 2011-12-31)
Missing months in range: []


In [46]:
# So sánh date range giữa các sheet
sheet_dates = pd.DataFrame(
    [
        {"sheet": name, "min_date": f[DATE_COL].min(), "max_date": f[DATE_COL].max(), "rows": len(f)}
        for name, f in raw_sheets.items()
    ]
)
sheet_dates

,sheet,min_date,max_date,rows
0,Year 2009-2010,2009-12-01 07:45:00,2010-12-09 20:01:00,525461
1,Year 2010-2011,2010-12-01 08:26:00,2011-12-09 12:50:00,541910


In [47]:
# Kiểm tra các sheet có chồng lấn thời gian không (sheet sau bắt đầu trước khi sheet trước kết thúc)
ordered = sheet_dates.sort_values("min_date").reset_index(drop=True)
for i in range(1, len(ordered)):
    prev, curr = ordered.loc[i - 1], ordered.loc[i]
    overlap = curr["min_date"] <= prev["max_date"]
    print(f"{prev['sheet']!r} vs {curr['sheet']!r} -> overlap: {overlap}")
    if overlap:
        start, end = curr["min_date"], prev["max_date"]
        in_prev = raw_sheets[prev["sheet"]][DATE_COL].between(start, end).sum()
        in_curr = raw_sheets[curr["sheet"]][DATE_COL].between(start, end).sum()
        print(f"   Overlap window: {start} -> {end}")
        print(f"   Rows in window: {prev['sheet']!r}={in_prev}, {curr['sheet']!r}={in_curr}")

'Year 2009-2010' vs 'Year 2010-2011' -> overlap: True
   Overlap window: 2010-12-01 08:26:00 -> 2010-12-09 20:01:00
   Rows in window: 'Year 2009-2010'=22523, 'Year 2010-2011'=22523


# Initial findings

> Tóm tắt dựa trên output thực tế khi chạy notebook (kernel Python 3.11, pandas 3.0.5).
> Nếu dataset raw thay đổi, cần chạy lại notebook và cập nhật các số liệu bên dưới.

### Dataset structure
- File có **2 sheet**: `Year 2009-2010` (525,461 × 8) và `Year 2010-2011` (541,910 × 8).
- Hai sheet có cùng schema 8 cột: `Invoice`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `Price`, `Customer ID`, `Country`.
- DataFrame audit tổng hợp: **1,067,371 rows** × 8 cột gốc (+ `source_sheet`).

### Data types
- `Invoice` và `StockCode` là `object` lẫn `int` và `str` (Invoice: 1,047,871 int / 19,500 str; StockCode: 932,385 int / 134,986 str).
- `Description` có 4 giá trị kiểu `int`.
- `Customer ID` bị đọc thành `float64` do có missing; mọi giá trị non-null đều là số nguyên.
- `InvoiceDate` đã là `datetime64` (pandas 3 hiển thị `datetime64[us]`), không có missing.

### Missing values
- `Customer ID`: **243,007 missing (22.767%)** — sheet 2009-2010: 20.539%, sheet 2010-2011: 24.927%.
- `Description`: **4,382 missing (0.411%)** — toàn bộ các dòng này cũng thiếu `Customer ID` và có `Price = 0`.
- `Invoice`, `StockCode`, `Quantity`, `InvoiceDate`, `Price`, `Country` không có missing.

### Duplicate records
- Duplicate toàn bộ row (trên 8 cột gốc): **34,335 bản sao dư (3.217%)**, 67,242 dòng thuộc các nhóm trùng.
- Trong từng sheet: 6,865 (1.306%) ở 2009-2010 và 5,268 (0.972%) ở 2010-2011.
- 22,202 bản trùng chỉ xuất hiện khi ghép 2 sheet → khớp với giai đoạn hai sheet chồng lấn (xem Date coverage).
- Dataset không có line-level ID nên duplicate trong cùng sheet chưa chắc là lỗi nghiệp vụ.

### Invalid values
- `Quantity < 0`: 22,950 rows (2.150%); không có `Quantity == 0`.
- `Price == 0`: 6,202 rows (0.581%); `Price < 0`: 5 rows (đều là `Adjust bad debt`, invoice prefix `A`, StockCode `B`).
- Giá trị cực trị: `|Quantity|` max = 80,995 (cặp invoice `581483` / `C581484` cùng khách hàng), 74,215 (`541431` / `C541433`); 941 rows có `|Quantity|` > p99.9 (600).
- `Price` max = 38,970; 1,068 rows có Price > p99.9 (217.016). Các giá lớn nhất thuộc mã phi sản phẩm (`M`/Manual, `BANK CHARGES`, `AMAZONFEE`).
- 6,093 rows / 62 stock code không bắt đầu bằng 5 chữ số, gồm nhiều mã phi sản phẩm: `POST`, `DOT`, `M`, `C2`, `D`, `S`, `BANK CHARGES`, `ADJUST`, `AMAZONFEE`, gift voucher, `TEST001`... (một số mã như `DCGS...` có vẻ vẫn là sản phẩm → cần review thủ công).
- Không có Description là chuỗi rỗng.

### Returns / cancellations
- Prefix invoice quan sát được: số thuần (1,047,871 rows), `C` (19,494 rows) và `A` (6 rows).
- Invoice `C`: **8,292 invoice (15.462% số invoice unique)**, **19,494 rows (1.826% số rows)**.
- 19,493 / 19,494 dòng `C` có Quantity âm; 1 dòng `C` có Quantity dương.
- 3,457 dòng Quantity âm **không** thuộc invoice `C`: tất cả có `Price = 0` và thiếu `Customer ID` (Description thường NaN hoặc kiểu "damaged") → nhiều khả năng là điều chỉnh kho, không phải trả hàng của khách.

### Date coverage
- `InvoiceDate` từ **2009-12-01 07:45** đến **2011-12-09 12:50**: 739 ngày lịch, 604 ngày có giao dịch.
- Records theo năm: 2009 = 45,228; 2010 = 522,714; 2011 = 499,429.
- Không thiếu tháng nào trong khoảng; tháng **2011-12 chỉ có dữ liệu đến ngày 09** (8 ngày giao dịch) → tháng cuối không đầy đủ.
- Hai sheet **chồng lấn** từ 2010-12-01 08:26 đến 2010-12-09 20:01 (22,523 rows trong mỗi sheet) → số record tháng 2010-12 trong bản ghép bị đếm trùng.

### Data quality issues
- Chồng lấn dữ liệu giữa 2 sheet gây duplicate khi concat.
- Kiểu dữ liệu không đồng nhất ở `Invoice`, `StockCode`, `Description`, `Customer ID`.
- Tỷ lệ thiếu `Customer ID` cao (~22.8%).
- Có cancellation, điều chỉnh kho, bad-debt adjustment và mã phí/phi sản phẩm lẫn trong giao dịch bán hàng.
- Có outlier Quantity/Price lớn, một số là cặp mua – hủy triệt tiêu nhau.

### Những vấn đề cần xử lý trong cleaning stage
1. Xử lý phần chồng lấn 2010-12-01 → 2010-12-09 khi ghép 2 sheet (loại bản trùng giữa sheet).
2. Quyết định chính sách cho duplicate trong cùng sheet (giữ hay loại) và ghi lại lý do.
3. Chuẩn hoá kiểu: `Invoice`, `StockCode` → string; `Customer ID` → `Int64`/string; `Description` → string.
4. Tách hoặc gắn cờ invoice `C` (cancellation), invoice `A` (bad debt adjustment) và Quantity âm không phải `C`.
5. Định nghĩa danh sách mã phi sản phẩm (`POST`, `DOT`, `M`, `D`, `BANK CHARGES`, `AMAZONFEE`, ...) để loại khỏi phân tích sản phẩm.
6. Xử lý dòng `Price <= 0` và dòng thiếu `Description`.
7. Chọn cách xử lý dòng thiếu `Customer ID` (giữ cho phân tích doanh thu, loại cho phân tích theo khách hàng).
8. Ghi chú tháng 2011-12 không đầy đủ khi tính chỉ số theo tháng.
9. Xem xét chính sách outlier (ví dụ cặp 80,995 / -80,995) thay vì xoá tự động.

*Notebook này chỉ audit và ghi nhận vấn đề; không xoá, fill, convert hay ghi đè dữ liệu raw.*